# MSE-Based Loss Functions

Fitting to summary statistics via simulation


`MemorySearchMseLoss` applies ordinary dataset analyses to observed and
simulated recalls. Each candidate is simulated once, then those responses are
reused across all configured analyses. Simulation uses each trial's own model;
trial metadata is repeated with the simulation rows, and item responses are
converted back to study-position codes before analysis.

The analysis determines trial/participant aggregation and the meaning of each
summary bin. Nonfinite observed bins are omitted. A missing simulated prediction
for an observed bin gives infinite loss. Analyses with no observed usable bins
are invalid unless disabled with zero weight.


```python
from jaxcmr.analyses.spc import spc
from jaxcmr.loss.mse import make_factory

# Pass this configured class as loss_fn_cls to an ordinary fitter.
loss_factory = make_factory({"spc": spc}, simulation_count=20, seed=0)
```

Analyses accept one `RecallDataset`, matching their normal library interface.
For example, a custom summary can read `dataset["recalls"]` and any aligned
study metadata directly. It receives observations and simulations in the same
representation; it needs no special simulation-array arguments.


## Variants

### SPC MSE

**Module:** `jaxcmr.loss.mse`

Fits to the **Serial Position Curve** (probability of recall by presentation position).

```python
from jaxcmr.loss.mse import MemorySearchSpcMseLoss
```


The SPC preset calls `jaxcmr.analyses.spc.spc`. Duplicate responses count once;
recalling an item presented repeatedly credits its study occurrences. Padding
and position-specific denominators follow the analysis implementation.

`MemorySearchSpcMseLoss` uses 20 simulations per trial and seed zero by default.
Both settings are configurable keyword arguments. Use `functools.partial` to
bind them when passing the constructor to a fitter.


### Category SPC MSE

**Module:** `jaxcmr.loss.mse`

Fits to **category-filtered** Serial Position Curves—separate SPCs for each stimulus category.

```python
from jaxcmr.loss.mse import MemorySearchCatSpcMseLoss
```


The category preset calls `cat_spc` for each requested category and stacks the
results into one summary. Its denominator at each position is the number of
eligible presentations of that category at that position. An absent category
contributes undefined bins, which are omitted while other categories remain usable.

Defaults are `category_field="condition"`, `category_values=(1, 2)`,
`simulation_count=10`, and `seed=0`. Categories annotate study positions.


`make_factory` also accepts analysis weights and fixed residual scales.
`reduction="pooled"` divides weighted squared errors by the weighted number of
observed usable bins. `reduction="per_analysis"` first averages each analysis's
squared errors, then takes their weighted mean. These choices can weight
analyses differently when they return different numbers of bins.

`component_scores` reports contributions that sum to the configured objective.
Their interpretation depends on the supplied analyses and scales.


```python
import jax.numpy as jnp
from jaxcmr.loss.mse import make_factory

def mean_response_count(dataset):
    return jnp.mean(jnp.count_nonzero(dataset["recalls"], axis=1))

loss_factory = make_factory({"response_count": mean_response_count}, seed=0)
```

This example counts recorded response slots. If the scientific measure should
count distinct recalled items or study positions, use an analysis implementing
that definition explicitly.


## Simulation and search

For a fixed dataset, count and seed, the same parameter values receive the same
simulation keys and objective. Changing the seed can change the approximation;
more simulations usually reduce sampling variation at additional cost.

Stopping remains active during simulation. Responses after the observed response
count are masked, so this objective does not simply force that many recalls.
Discrete categorical sampling does not supply useful pathwise gradients. Use a
population search such as differential evolution rather than treating the MSE
objective as a smooth gradient loss.

## SPC fitting example

Configure the observation file, model and bounds for your experiment before
running this example. Search settings are illustrative, not convergence evidence.


In [ ]:
from jaxcmr.dataset import generate_trial_mask
from jaxcmr.helpers import load_data

data_path = "experiment.h5"  # Set this to your recall-data file.
data = load_data(data_path)
trial_mask = generate_trial_mask(data, None)

from jaxcmr.components.context import init as init_context
from jaxcmr.components.linear_memory import init_mcf, init_mfc
from jaxcmr.components.termination import PositionalTermination
from jaxcmr.fitting import ScipyDE
from jaxcmr.loss.mse import MemorySearchSpcMseLoss
from jaxcmr.models.cmr import make_factory

# Create model factory
model_factory_cls = make_factory(
    init_mfc, init_mcf, init_context, PositionalTermination
)

# Define parameters
fixed_params = {
    "allow_repeated_recalls": False,
    "learn_after_context_update": True,
    "learning_rate": 0.5,
    "shared_support": 0.05,
    "item_support": 0.25,
    "choice_sensitivity": 0.6,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
}

free_params = {
    "encoding_drift_rate": [0.0, 1.0],
    "start_drift_rate": [0.0, 1.0],
    "recall_drift_rate": [0.0, 1.0],
    "primacy_scale": [0.0, 10.0],
    "primacy_decay": [0.0, 5.0],
}

# Fit to SPC shape
fitter = ScipyDE(
    dataset=data,
    features=None,
    base_params=fixed_params,
    model_factory_cls=model_factory_cls,
    loss_fn_cls=MemorySearchSpcMseLoss,
    hyperparams={
        "bounds": free_params,
        "num_steps": 1500,  # Illustrative search limit
        "best_of": 3,
    },
)

results = fitter.fit(trial_mask)

## Inspect predicted behavior

The following simulation is an additional predictive check. Its default trial
procedure generates recall freely; it does not apply the MSE loss's observed-count
mask. Comparing these predictions can reveal behavior that the chosen objective
leaves unconstrained.


In [ ]:
import jax.numpy as jnp
from jax import random

from jaxcmr.fitting import parameters_for_subjects

key = random.PRNGKey(0)
fitted_params = parameters_for_subjects(results, jnp.unique(data["subject"].ravel()[trial_mask]))
from jaxcmr.analyses.spc import plot_spc
from jaxcmr.simulation import simulate_dataset

# Simulate from fitted parameters
sim = simulate_dataset(
    model_factory_cls, data, None,
    fitted_params, trial_mask,
    experiment_count=100, rng=key,
)

# Compare
plot_spc(
    datasets=[data, sim],
    trial_masks=[trial_mask, generate_trial_mask(sim, None)],
    labels=["Data", "Model"],
)

## Interpretation

A small MSE indicates agreement for the selected summaries and scales. It is
not a log likelihood and cannot be inserted into likelihood-based AIC/BIC or
likelihood-ratio calculations. Examine other measures and held-out predictions
when assessing the model more broadly.

Old SPC/category-SPC MSE scores used different counting and simulation conventions;
recalculate them before comparing with the corrected definitions. Existing saved
fits and research outputs are not rewritten.

See the [MSE API reference](../../../guide/reference/loss-mse.qmd).
